In [1]:
import numpy as np
import sympy as sp
import jax 
import jax.numpy as jnp

calculate dot product

In [2]:
# Define the matrix A and vector u
A = np.array([[1, 2], [3, 4]])
u = np.array([[1], [-1]])

np.dot(u.T, np.dot(A, u))

array([[0]])

compute eigenvalue

In [6]:
A = np.array([[0,1],[-2,-3]])
eigenval = np.linalg.eigvals(A)
eigenvalues, eigenvectors = np.linalg.eig(A)

In [24]:
# generate matrix using known eigenvalues
eigenvalues_A = np.array([2,3])
eigenvalues_B = np.array([100, 200])

D_A = np.diag(eigenvalues_A)
D_B = np.diag(eigenvalues_B)  

P, _ = np.linalg.qr(np.random.randn(2,2)) # invertible orthogonal matrix
invertible_P = np.array([[2,1],[1,3]])

A = invertible_P @ D_A @ np.linalg.inv(invertible_P)
B = invertible_P @ D_B @ np.linalg.inv(invertible_P)

eigenvalue_new = np.linalg.eigvals(A+B)
eigenvalue_new
#np.linalg.det(invertible_P) compute determinant

array([102., 203.])

Generate Matrix

In [ ]:
def create_matrix(n):
    return np.fromfunction(lambda i,j: i+j+2, (n,n))

compute first derivative

In [28]:
def f(x):
    return jnp.linalg.norm(x)

x = jnp.array([1.0,2.0])
grad_f = jax.grad(f)(x)
grad_f

Array([0.4472136, 0.8944272], dtype=float32)

compute second derivative

In [13]:
# Define the variable
x = sp.symbols('x', real=True, positive=True)

# Define the function
F = x * sp.log(x)

# Compute the second derivative
second_derivative = sp.diff(F, x, 2)

second_derivative

1/x

comput hessian

In [12]:
x, y = sp.symbols('x y')

# Define the function
F = sp.sin(x + 2*y)

# Calculate the Hessian matrix
Hessian = sp.hessian(F, (x, y))
Hessian

Matrix([
[  cos(x + 2*y)],
[2*cos(x + 2*y)]])

In [12]:
x, y = sp.symbols('x y')

F = x**2 + sp.sin(x)/10

Hessian = sp.hessian(F, (x, y))
Hessian

Matrix([
[2 - sin(x)/10, 0],
[            0, 0]])

In [29]:
def f(x):
    return jnp.log(2+jnp.sin(x[0]*x[1]))

x = jnp.array([1.,1.])
hess_f = jax.hessian(f)
hess_f(x)

Array([[-0.33229578, -0.14214697],
       [-0.14214697, -0.33229578]], dtype=float32)

find global minimizer

In [16]:
from scipy.optimize import minimize
def f(x):
    return (1 + np.cos(x[0])) + 3 * (x[0] + np.pi)**2

x_init = np.array([0])
minimize(f,x_init, method='BFGS')

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 1.3013855002302808e-16
        x: [-3.142e+00]
      nit: 5
      jac: [ 5.186e-09]
 hess_inv: [[ 1.429e-01]]
     nfev: 12
     njev: 6

In [20]:
import numpy as np
from scipy.optimize import minimize

def cross_entropy(y, x):
    # Ensure no zero probabilities; this is necessary to avoid log(0)
    y = np.clip(y, 1e-15, 1 - 1e-15)
    return -np.sum(x * np.log(y))

# Fixed x values (usually your data or target distribution)
x = np.array([0.25, 0.25, 0.25, 0.25])

# Initial guess for y
y_init = np.array([0.1, 0.2, 0.3, 0.4])

# Constraint that sum of y must be 1
cons = ({
    'type': 'eq',
    'fun': lambda y: np.sum(y) - 1
})

# Bounds for y to ensure they are probabilities
bounds = [(0, 1) for _ in y_init]

# Perform the minimization with constraints
result = minimize(cross_entropy, y_init, args=(x,), method='SLSQP', constraints=cons, bounds=bounds)

# Print the results
print("Optimal y:", result.x)
print("Minimum cross-entropy:", result.fun)

Optimal y: [0.24988937 0.24976379 0.25021192 0.25013493]
Minimum cross-entropy: 1.3862946234266627


Diagonalization

In [4]:
def is_diagonalizable(A):
    eigenvalues, eigenvectors = np.linalg.eig(A)
    print(eigenvalues)
    return np.linalg.matrix_rank(eigenvectors) == A.shape[0]

In [9]:
def diagonalize_matrix(A):
    # Ensure A is a square matrix
    if A.shape[0] != A.shape[1]:
        raise ValueError("Matrix must be square to be diagonalizable.")

    # Compute eigenvalues and eigenvectors
    eigenvalues, eigenvectors = np.linalg.eig(A)

    # Form the diagonal matrix from the eigenvalues
    D = np.diag(eigenvalues)

    # The matrix of eigenvectors, V, and its inverse
    V = eigenvectors
    V_inv = np.linalg.inv(V)

    # Verify if diagonalization is valid (V must be invertible)
    if not np.allclose(np.dot(V, V_inv), np.eye(A.shape[0])):
        raise ValueError("Matrix is not diagonalizable.")

    return V, D, V_inv


Matrix V (Eigenvectors):
[[0.89442719 0.70710678]
 [0.4472136  0.70710678]]
Diagonal matrix D (Eigenvalues):
[[3. 0.]
 [0. 2.]]
Inverse of V:
[[ 2.23606798 -2.23606798]
 [-1.41421356  2.82842712]]


In [10]:
 A = np.array(
  [
   [4,2],
   [0,1]
  ]
 )

is_diagonalizable(A)

diagonalize_matrix(A)

[4. 1.]


(array([[ 1.        , -0.5547002 ],
        [ 0.        ,  0.83205029]]),
 array([[4., 0.],
        [0., 1.]]),
 array([[1.        , 0.66666667],
        [0.        , 1.20185043]]))

Convexity check

In [31]:
def is_convex(*args, f):
    hess_f = sp.hessian(f, args)
    eigenvals = hess_f.eigenvals()
    return all((eigenval.is_real and eigenval >= 0) for eigenval in eigenvals)

""" x,y = sp.symbols('x y')
f = x**2 + y**2
is_convex(x,y,f = f) """

True

PD PSD check

In [32]:
def is_psd(A):
    return np.all(np.linalg.eigvalsh(A) >= 1e-10)

In [51]:
def is_pd(A):
    return np.all(np.linalg.eigvalsh(A) > 1e-10)

In [47]:
# compute p(M=RR^T is pd)
N = 10000
pd_count = 0
for i in range(N):
    R = np.random.normal(0,1,size=(5,5))
    M = R @ R.T
    pd_count += is_pd(M)

pd_count/N

1.0

In [58]:
# compute p(M=0.5(R+R^T) is pd)
N = 10000
pd_count = 0
for i in range(N):
    R = np.random.normal(0,1,size=(5,5))
    M = 0.5*(R + R.T)
    pd_count += is_pd(M)

pd_count/N

0.0001

Simulate Limit

Calculate output size

In [14]:
def calculate_output_size(H_in, W_in, kernel_height, kernel_width, stride_height=1, stride_width=1, pad_height=0, pad_width=0, dilation_height=1, dilation_width=1):
    H_out = (H_in + 2 * pad_height - dilation_height * (kernel_height - 1) - 1) // stride_height + 1
    W_out = (W_in + 2 * pad_width - dilation_width * (kernel_width - 1) - 1) // stride_width + 1
    return H_out, W_out

In [15]:
calculate_output_size(10,12,2,2,stride_height=2,stride_width=2)

(5, 6)

Compute number of parameter of a fully connected neural network

In [21]:
def calculate_parameters(n_hidden_layers, neurons_per_layer, input_dim, output_dim):
    # Parameters for the first layer (input to first hidden)
    total_params = input_dim * neurons_per_layer + neurons_per_layer  # Weights + biases

    # Parameters for each hidden layer
    for _ in range(n_hidden_layers - 1):
        total_params += neurons_per_layer * neurons_per_layer  # Weights between layers
        total_params += neurons_per_layer  # Biases for each neuron

    # Parameters for the output layer
    total_params += neurons_per_layer * output_dim  # Weights from last hidden to output
    total_params += output_dim  # Biases at output layer

    return total_params

In [22]:
calculate_parameters(3,10,3,1)

271